In [ ]:
import os
from pathlib import Path


def find_repo_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "downstream_tasks/expression_prediction").is_dir():
            return path
    raise RuntimeError("Run inside the GENA-LM clone or set GENA_HOME")


REPO_ROOT = (
    Path(os.environ["GENA_HOME"]).resolve()
    if "GENA_HOME" in os.environ
    else find_repo_root()
)
BENCHMARK_ROOT = Path(os.environ.get("BENCHMARK_ROOT", REPO_ROOT)).resolve()
TASK_ROOT = Path(os.environ.get("TASK_ROOT", REPO_ROOT)).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", REPO_ROOT / "data")).resolve()


# GENA-LM mouse benchmark

Compare all mouse predictions with the same 243-track ground-truth matrix.

The coordinate files `mouse.{valid,test}.{forward,reverse}.csv` contain gene positions only. They are not expression ground truth.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
BASE = BENCHMARK_ROOT
PRED_ROOT = TASK_ROOT / "predictions_results_mm10"

GT_PATH = DATA_ROOT / "true_mouse_all_genes_qnorm_samples_by_genes.WITH_TEST.csv"
OUTPUT_PATH = BASE / "graund_true_comparison/results/gena_lm_mm10_gt_summary.csv"

MODEL_DIRS = {
    "dev_loss": PRED_ROOT / "dev_loss",
    "all_datasets_2": PRED_ROOT / "all_datasets2",
    "all_datasets": PRED_ROOT / "all_datasets",
    "glioma": PRED_ROOT / "glioma",
    "len_2048": PRED_ROOT / "len_2048",
    "ATAC": PRED_ROOT / "ATAC",
    "mult_loss": PRED_ROOT / "mult_loss",
    "xlarge": PRED_ROOT / "xlarge",
}
MODEL_ORDER = list(MODEL_DIRS)
SPLITS = ["valid", "test"]


In [ ]:
def load_gene_by_cell(path):
    df = pd.read_csv(path)

    # genes x cell IDs
    if "gene_id" in df.columns and any(c.startswith("ENCFF") for c in df.columns):
        out = df.set_index("gene_id")
        out = out[[c for c in out.columns if c.startswith("ENCFF")]]

    # cell IDs x genes
    elif "id" in df.columns and any(c.startswith("ENSMUSG") for c in df.columns):
        genes = [c for c in df.columns if c.startswith("ENSMUSG")]
        out = df.set_index("id")[genes].T

    else:
        raise ValueError("GT must contain gene_id + ENCFF columns, or id + ENSMUSG columns")

    out.index = out.index.astype(str)
    return out.astype(float)


def pairwise_corr(x, y, axis):
    valid = np.isfinite(x) & np.isfinite(y)
    n = valid.sum(axis=axis)
    xx = np.where(valid, x, 0.0)
    yy = np.where(valid, y, 0.0)
    sx, sy = xx.sum(axis=axis), yy.sum(axis=axis)
    sxx, syy = (xx * xx).sum(axis=axis), (yy * yy).sum(axis=axis)
    sxy = (xx * yy).sum(axis=axis)
    numerator = n * sxy - sx * sy
    denominator = np.sqrt((n * sxx - sx**2) * (n * syy - sy**2))
    return np.divide(numerator, denominator, out=np.full(n.shape, np.nan, dtype=float),
                     where=(n > 3) & (denominator > 0))


def correlations(gt, pred):
    genes = gt.index.intersection(pred.index)
    cells = gt.columns.intersection(pred.columns)
    x = gt.loc[genes, cells].to_numpy(float)
    y = pred.loc[genes, cells].to_numpy(float)
    gene_corr = np.nanmean(pairwise_corr(x, y, axis=0))  # across genes, then mean cells
    cell_corr = np.nanmean(pairwise_corr(x, y, axis=1))  # across cells, then mean genes
    return float(gene_corr), float(cell_corr), len(genes), len(cells)


In [ ]:
if not GT_PATH.exists():
    raise FileNotFoundError(
        f"Mouse expression GT is missing: {GT_PATH}\n"
        "The existing mouse.valid/test.forward/reverse.csv files contain coordinates only."
    )

# Same transformation used in the human benchmark.
gt = np.log2(load_gene_by_cell(GT_PATH) + 1)
print("mouse GT:", gt.shape)
if gt.shape[1] != 243:
    print(f"WARNING: expected 243 GT tracks, found {gt.shape[1]}")


In [ ]:
rows = []

for checkpoint, model_dir in MODEL_DIRS.items():
    for split in SPLITS:
        path = model_dir / f"gena_lm_{split}_json243_mm10_predictions.csv"
        row = {"checkpoint": checkpoint, "split": split}
        try:
            pred = pd.read_csv(path).set_index("gene_id").astype(float)
            gene_corr, cell_corr, n_genes, n_cells = correlations(gt, pred)
            row.update(gene_corr=gene_corr, cell_corr=cell_corr,
                       n_genes=n_genes, n_cells=n_cells, status="ok")
        except Exception as error:
            row.update(gene_corr=np.nan, cell_corr=np.nan,
                       n_genes=0, n_cells=0, status=str(error))
        rows.append(row)
        print(checkpoint, split, row["status"])

results_long = pd.DataFrame(rows)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_long.to_csv(OUTPUT_PATH, index=False)
print("saved:", OUTPUT_PATH)


In [ ]:
summary = results_long.pivot(
    index="checkpoint",
    columns="split",
    values=["gene_corr", "cell_corr"],
).reorder_levels([1, 0], axis=1)

columns = pd.MultiIndex.from_product(
    [["valid", "test"], ["gene_corr", "cell_corr"]],
    names=["split", "metric"],
)
summary = summary.reindex(index=MODEL_ORDER, columns=columns)
summary.columns = pd.MultiIndex.from_tuples(
    [(f"mouse {split} (243 cell types)", metric) for split, metric in summary.columns],
    names=["benchmark", "metric"],
)
display(summary.round(3))
summary.to_csv(OUTPUT_PATH.with_name("gena_lm_mm10_gt_table.csv"), float_format="%.3f")


In [ ]:
OUTPUT_PATH

In [ ]:
problems = results_long.loc[results_long.status != "ok", ["checkpoint", "split", "status"]]
display(problems if not problems.empty else pd.DataFrame({"status": ["All runs completed"]}))
